# Experiment 2 — Word2Vec

Trong phần này, Word2Vec được huấn luyện trên corpus C4 khoảng 30.000 documents đã dùng ở LAB 02. Notebook chuẩn bị câu đã tokenize, huấn luyện model bằng Gensim, rồi kiểm tra vocabulary và embedding. Experiment 3/4 bên dưới khảo sát ảnh hưởng của context window và embedding dimension.

## 1. Import libraries

Dùng regex và thư viện chuẩn của Python để đọc, tách câu và tokenize. Gensim chỉ được dùng để train Word2Vec.

In [1]:
import json
import re
from pathlib import Path

from gensim.models import Word2Vec

print("Gensim Word2Vec đã sẵn sàng.")

Gensim Word2Vec đã sẵn sàng.


## 2. Load corpus

Corpus là các documents từ file JSON Lines của C4. Mỗi dòng JSON có trường text. Notebook đọc tối đa 30.000 documents.

In [2]:
DATA_FILENAME = "c4-train.00000-of-01024-30K.json"
MAX_DOCUMENTS = 30_000

candidate_paths = [
    Path.cwd() / DATA_FILENAME,
    Path.cwd() / "Lab3" / DATA_FILENAME,
    Path.cwd().parent / "Lab3" / DATA_FILENAME,
    Path.cwd() / "NLP" / "Lab3" / DATA_FILENAME,
    Path("/home/nguyenducthang/NLP/Lab3") / DATA_FILENAME,
    Path("/content/NLP/Lab3") / DATA_FILENAME,
]

DATA_PATH = next((path for path in candidate_paths if path.is_file()), None)

if DATA_PATH is None:
    checked = "\n".join(f"- {path}" for path in candidate_paths)
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_FILENAME}. Đã kiểm tra các vị trí:\n{checked}"
    )

documents = []
with DATA_PATH.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if len(documents) >= MAX_DOCUMENTS:
            break
        if not line.strip():
            continue

        record = json.loads(line)
        text = record.get("text", "") if isinstance(record, dict) else ""
        documents.append(text if isinstance(text, str) else "")

print("Corpus file:", DATA_PATH)
print("Số documents đã nạp:", len(documents))
print("Document mẫu:")
print(documents[0][:500] if documents else "(corpus rỗng)")

Corpus file: /home/nguyenducthang/NLP/Lab3/c4-train.00000-of-01024-30K.json
Số documents đã nạp: 30000
Document mẫu:
Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.
He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat select


## 3. Preprocess: sentence and word tokenization

Chuyển chữ thường, tách câu theo dấu kết thúc câu hoặc xuống dòng, rồi lấy các token chữ/số. Mỗi câu được giữ thành một list token, đúng định dạng đầu vào của Gensim Word2Vec.

In [3]:
SENTENCE_SPLIT_PATTERN = re.compile(r"(?<=[.!?])\s+|\n+")
WORD_PATTERN = re.compile(r"[a-z0-9]+(?:['’][a-z0-9]+)*")

tokenized_sentences = []

for document in documents:
    text = document.lower()
    sentence_chunks = SENTENCE_SPLIT_PATTERN.split(text)

    for chunk in sentence_chunks:
        tokens = WORD_PATTERN.findall(chunk)
        if tokens:
            tokenized_sentences.append(tokens)

# Bỏ bản raw text để giảm RAM sau khi đã tokenize xong.
del documents

print("Corpus file:", DATA_PATH)
print("Số câu đã tokenize:", len(tokenized_sentences))
print("Tổng số token:", sum(len(sentence) for sentence in tokenized_sentences))
print("Một vài câu sau tokenize:")
for sentence in tokenized_sentences[:5]:
    print(sentence)

Corpus file: /home/nguyenducthang/NLP/Lab3/c4-train.00000-of-01024-30K.json
Số câu đã tokenize: 634941
Tổng số token: 10972642
Một vài câu sau tokenize:
['beginners', 'bbq', 'class', 'taking', 'place', 'in', 'missoula']
['do', 'you', 'want', 'to', 'get', 'better', 'at', 'making', 'delicious', 'bbq']
['you', 'will', 'have', 'the', 'opportunity', 'put', 'this', 'on', 'your', 'calendar', 'now']
['thursday', 'september', '22nd', 'join', 'world', 'class', 'bbq', 'champion', 'tony', 'balay', 'from', 'lonestar', 'smoke', 'rangers']
['he', 'will', 'be', 'teaching', 'a', 'beginner', 'level', 'class', 'for', 'everyone', 'who', 'wants', 'to', 'get', 'better', 'with', 'their', 'culinary', 'skills']


## 4. Word2Vec configuration

Cấu hình theo yêu cầu. Gensim dùng `sg=0` cho CBOW và `sg=1` cho Skip-gram. Ở đây chọn `sg=0`.

In [4]:
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 2
EPOCHS = 10
SG = 0
SEED = 42

if SG == 0:
    TRAINING_OBJECTIVE = "CBOW (dự đoán target word từ các context words)"
else:
    TRAINING_OBJECTIVE = "Skip-gram (dùng target word để dự đoán context words)"

print("Corpus:", DATA_PATH.name)
print("Số câu:", len(tokenized_sentences))
print("vector_size:", VECTOR_SIZE)
print("window:", WINDOW)
print("min_count:", MIN_COUNT)
print("epochs:", EPOCHS)
print("sg:", SG)
print("Training objective:", TRAINING_OBJECTIVE)

Corpus: c4-train.00000-of-01024-30K.json
Số câu: 634941
vector_size: 100
window: 5
min_count: 2
epochs: 10
sg: 0
Training objective: CBOW (dự đoán target word từ các context words)


## 5. Train Word2Vec

Train model từ corpus đã tokenize. `workers=1` và seed cố định giúp kết quả có tính lặp lại tốt hơn.

In [5]:
model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=1,
    sg=SG,
    seed=SEED,
    epochs=EPOCHS,
)

print("Word2Vec training hoàn tất.")

Word2Vec training hoàn tất.


## 6. Inspect vocabulary and one embedding

Vocabulary dưới đây là vocabulary sau khi Gensim áp dụng `min_count`. In kích thước, một số token đầu, kiểm tra vài từ có trong vocabulary hay không, và xem một vector mẫu.

In [6]:
model_vocabulary = model.wv.index_to_key
print("Vocabulary size của model:", len(model_vocabulary))
print("Một số từ trong vocabulary:", model_vocabulary[:20])

words_to_check = ["cat", "dog", "eats", "likes", "fish", "milk", "meat"]
for word in words_to_check:
    print(f"{word}: {'có' if word in model.wv else 'không có'}")

if "the" in model.wv:
    sample_word = "the"
elif model_vocabulary:
    sample_word = model_vocabulary[0]
else:
    raise ValueError("Model vocabulary rỗng; hãy kiểm tra corpus và min_count.")

sample_vector = model.wv[sample_word]
print("Từ mẫu:", sample_word)
print("Kích thước vector:", sample_vector.shape)
print("10 giá trị đầu của vector:", sample_vector[:10])

Vocabulary size của model: 102380
Một số từ trong vocabulary: ['the', 'and', 'to', 'of', 'a', 'in', 'is', 'for', 'you', 'that', 'with', 'on', 'it', 'i', 'are', 'as', 'this', 'be', 'your', 'or']
cat: có
dog: có
eats: có
likes: có
fish: có
milk: có
meat: có
Từ mẫu: the
Kích thước vector: (100,)
10 giá trị đầu của vector: [-0.26430312  1.599196    1.662482    0.22267182 -0.06629112  1.3269558
 -0.5665084  -1.3004221   0.2343602   0.14358158]


## Tóm tắt cấu hình

- **Corpus:** C4 JSON Lines, tối đa 30.000 documents; các documents được tách thành câu đã tokenize.
- **Vocabulary:** các token đủ tần suất sau khi Gensim áp dụng `min_count=2`.
- **vector_size = 100:** mỗi từ được biểu diễn bằng vector 100 chiều.
- **window = 5:** model xét context tối đa 5 từ quanh vị trí target.
- **min_count = 2:** loại các từ xuất hiện ít hơn 2 lần khỏi vocabulary của model.
- **epochs = 10:** model đi qua corpus huấn luyện 10 lượt.
- **sg = 0, CBOW:** model dùng các context words để dự đoán target word.

### AI assistance statement

W3 yêu cầu khai báo nếu dùng AI. Hãy rà soát và điền trung thực phần đóng góp của bạn trước khi nộp:

- Tool: …
- Purpose: …
- Generated content: …
- Modified content: …
- Verification: …

# Experiment 3 — Context window

Dùng lại cùng corpus và preprocessing của Experiment 2. Ba model có cùng vector size, min_count, epochs, training objective và seed; chỉ thay đổi window. Kết quả similarity sẽ được tính từ model thực tế sau khi train.

## 1. Train ba model với window khác nhau

In [7]:
WINDOWS_TO_COMPARE = [2, 5, 10]
WORD_PAIRS = [
    ("doctor", "physician"),
    ("doctor", "hospital"),
    ("cat", "dog"),
]

# Tái dùng đúng tokenized_sentences và các hyperparameter từ Experiment 2.
models_by_window = {}
for current_window in WINDOWS_TO_COMPARE:
    print(f"Đang train Word2Vec với window={current_window}...")
    models_by_window[current_window] = Word2Vec(
        sentences=tokenized_sentences,
        vector_size=VECTOR_SIZE,
        window=current_window,
        min_count=MIN_COUNT,
        workers=1,
        sg=SG,
        seed=SEED,
        epochs=EPOCHS,
    )

print("Số câu dùng cho cả ba model:", len(tokenized_sentences))
print("vector_size:", VECTOR_SIZE)
print("min_count:", MIN_COUNT)
print("epochs:", EPOCHS)
print("sg:", SG, "—", TRAINING_OBJECTIVE)
print("window đã thử:", WINDOWS_TO_COMPARE)

Đang train Word2Vec với window=2...


Đang train Word2Vec với window=5...


Đang train Word2Vec với window=10...


Số câu dùng cho cả ba model: 634941
vector_size: 100
min_count: 2
epochs: 10
sg: 0 — CBOW (dự đoán target word từ các context words)
window đã thử: [2, 5, 10]


## 2. Kiểm tra vocabulary và tính cosine similarity

In [8]:
# Kiểm tra từng từ trước khi tính similarity để tránh KeyError.
similarity_records = []
similarities_by_window = {}

for current_window, current_model in models_by_window.items():
    print(f"\nwindow={current_window}; vocabulary size={len(current_model.wv)}")
    similarities_by_window[current_window] = {}

    for word_a, word_b in WORD_PAIRS:
        missing_words = [
            word for word in (word_a, word_b)
            if word not in current_model.wv
        ]

        if missing_words:
            similarity = None
            status = "missing from vocabulary: " + ", ".join(missing_words)
            print(f"{word_a} – {word_b}: không tính được ({status})")
        else:
            similarity = float(current_model.wv.similarity(word_a, word_b))
            status = "ok"
            print(f"{word_a} – {word_b}: {similarity:.6f}")

        pair_name = f"{word_a}–{word_b}"
        similarities_by_window[current_window][pair_name] = similarity
        similarity_records.append({
            "experiment": "Experiment 3 - Context window",
            "window": current_window,
            "word_pair": pair_name,
            "similarity": similarity,
            "status": status,
            "vector_size": VECTOR_SIZE,
            "min_count": MIN_COUNT,
            "epochs": EPOCHS,
            "sg": SG,
            "objective": TRAINING_OBJECTIVE,
        })


window=2; vocabulary size=102380
doctor – physician: 0.750459
doctor – hospital: 0.432198
cat – dog: 0.692171

window=5; vocabulary size=102380
doctor – physician: 0.753575
doctor – hospital: 0.450375
cat – dog: 0.773208

window=10; vocabulary size=102380
doctor – physician: 0.773821
doctor – hospital: 0.489994
cat – dog: 0.825869


## 3. Bảng tổng hợp và lưu results.csv

In [9]:
# Bảng: mỗi hàng là một word pair, mỗi cột là một window.
table_rows = [
    "| Word pair | Window 2 | Window 5 | Window 10 |",
    "| --- | ---: | ---: | ---: |",
]

for word_a, word_b in WORD_PAIRS:
    pair_name = f"{word_a}–{word_b}"
    values = []
    for current_window in WINDOWS_TO_COMPARE:
        value = similarities_by_window[current_window][pair_name]
        values.append("N/A" if value is None else f"{value:.6f}")
    table_rows.append(f"| {pair_name} | {values[0]} | {values[1]} | {values[2]} |")

from IPython.display import display, Markdown
display(Markdown("\n".join(table_rows)))

# Bổ sung kết quả Experiment 3 vào results.csv và giữ các dòng đã có.
import csv

RESULTS_PATH = DATA_PATH.parent / "results.csv"
new_fields = [
    "experiment", "window", "word_pair", "similarity", "status",
    "vector_size", "min_count", "epochs", "sg", "objective",
]

old_rows = []
old_fields = []
if RESULTS_PATH.exists():
    with RESULTS_PATH.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        old_fields = reader.fieldnames or []
        old_rows = list(reader)

# Thay các dòng Experiment 3 cũ để chạy lại cell không tạo bản ghi trùng.
old_rows = [
    row for row in old_rows
    if row.get("experiment") != "Experiment 3 - Context window"
]
fieldnames = list(dict.fromkeys(old_fields + new_fields))

with RESULTS_PATH.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(old_rows)
    writer.writerows(similarity_records)

print("Đã lưu kết quả vào:", RESULTS_PATH)
print("Số dòng kết quả Experiment 3:", len(similarity_records))

| Word pair | Window 2 | Window 5 | Window 10 |
| --- | ---: | ---: | ---: |
| doctor–physician | 0.750459 | 0.753575 | 0.773821 |
| doctor–hospital | 0.432198 | 0.450375 | 0.489994 |
| cat–dog | 0.692171 | 0.773208 | 0.825869 |

Đã lưu kết quả vào: /home/nguyenducthang/NLP/Lab3/results.csv
Số dòng kết quả Experiment 3: 9


### Dữ liệu quan sát được và câu hỏi tự trả lời

| Word pair | Window 2 | Window 5 | Window 10 |
| --- | ---: | ---: | ---: |
| doctor–physician | 0.750459 | 0.753575 | 0.773821 |
| doctor–hospital | 0.432198 | 0.450375 | 0.489994 |
| cat–dog | 0.692171 | 0.773208 | 0.825869 |

Trong cả ba cặp được đo, similarity tăng ở mỗi bước window 2 → 5 → 10. Đây là số liệu của ba cặp cụ thể trong lần chạy này.

**Câu hỏi:** “Window lớn hơn có luôn tốt hơn không?”  
**Câu trả lời của em:** Không. Window lớn hơn không phải lúc nào cũng tốt hơn.

**Lý do dựa trên kết quả và giới hạn của phép thử:** Trong lần chạy này, similarity của cả ba cặp từ đều tăng khi window tăng từ 2 → 5 → 10. Ví dụ, doctor–physician tăng từ 0.750459 lên 0.773821, doctor–hospital tăng từ 0.432198 lên 0.489994 và cat–dog tăng từ 0.692171 lên 0.825869. Điều này cho thấy với corpus và các cặp từ đang xét, window lớn hơn giúp model thu nhận được nhiều context hơn và làm các vector của các từ liên quan trở nên gần nhau hơn.

Tuy nhiên, chưa thể kết luận rằng window càng lớn thì luôn càng tốt, vì phép thử này chỉ sử dụng ba cặp từ và một corpus cụ thể. Khi window quá lớn, model có thể thu nhận thêm nhiều từ xa target và ít liên quan trực tiếp, làm context trở nên nhiễu. Vì vậy, kích thước window phù hợp còn phụ thuộc vào corpus, loại quan hệ ngữ nghĩa cần học và mục tiêu của bài toán.


# Experiment 4 — Embedding dimension

Dùng lại corpus và tokenized sentences của Experiment 2/3. So sánh vector_size=50, 100, 300; giữ cố định window=5, min_count, epochs, objective và seed. Kết quả số được tính từ model thực tế khi chạy các cell.


## 1. Train và đo training time/model size

In [10]:
from time import perf_counter

DIMENSIONS = [50, 100, 300]
DIMENSION_WINDOW = 5
DIMENSION_MODELS = {}
dimension_stats = {}

# Chỉ vector_size thay đổi; các thiết lập còn lại được giữ nguyên.
for dimension in DIMENSIONS:
    start_time = perf_counter()
    dimension_model = Word2Vec(
        sentences=tokenized_sentences,
        vector_size=dimension,
        window=DIMENSION_WINDOW,
        min_count=MIN_COUNT,
        workers=1,
        sg=SG,
        seed=SEED,
        epochs=EPOCHS,
    )
    training_seconds = perf_counter() - start_time
    DIMENSION_MODELS[dimension] = dimension_model

    # Đo ma trận input embeddings: số values và số bytes, không gồm các weights khác.
    embedding_values = int(dimension_model.wv.vectors.size)
    embedding_bytes = int(dimension_model.wv.vectors.nbytes)
    dimension_stats[dimension] = {
        "training_time_seconds": training_seconds,
        "embedding_values": embedding_values,
        "embedding_bytes": embedding_bytes,
        "vocabulary_size": len(dimension_model.wv),
    }
    print(
        f"dimension={dimension}: train={training_seconds:.2f}s, "
        f"vocabulary={len(dimension_model.wv)}, "
        f"embedding values={embedding_values:,}, "
        f"embedding matrix={embedding_bytes / (1024 ** 2):.2f} MiB"
    )

print("Fixed settings: window=5, min_count=", MIN_COUNT,
      ", epochs=", EPOCHS, ", sg=", SG, " (", TRAINING_OBJECTIVE, "), seed=", SEED)
print("Corpus sentences:", len(tokenized_sentences))


dimension=50: train=139.50s, vocabulary=102380, embedding values=5,119,000, embedding matrix=19.53 MiB


dimension=100: train=139.15s, vocabulary=102380, embedding values=10,238,000, embedding matrix=39.05 MiB


dimension=300: train=230.07s, vocabulary=102380, embedding values=30,714,000, embedding matrix=117.16 MiB
Fixed settings: window=5, min_count= 2 , epochs= 10 , sg= 0  ( CBOW (dự đoán target word từ các context words) ), seed= 42
Corpus sentences: 634941


## 2. Word-pair cosine similarity và analogy

In [11]:
DIMENSION_WORD_PAIRS = [
    ("doctor", "physician"),
    ("doctor", "hospital"),
    ("cat", "dog"),
]
ANALOGY = {"positive": ["king", "woman"], "negative": ["man"]}

dimension_similarity = {}
dimension_analogy = {}
dimension_result_rows = []

for dimension, dimension_model in DIMENSION_MODELS.items():
    wv = dimension_model.wv
    dimension_similarity[dimension] = {}
    print(f"\nDimension {dimension}; vocabulary size={len(wv)}")

    for word_a, word_b in DIMENSION_WORD_PAIRS:
        missing = [word for word in (word_a, word_b) if word not in wv]
        if missing:
            score = None
            status = "missing from vocabulary: " + ", ".join(missing)
            print(f"{word_a} – {word_b}: unavailable ({status})")
        else:
            score = float(wv.similarity(word_a, word_b))
            status = "ok"
            print(f"{word_a} – {word_b}: {score:.6f}")
        pair_name = f"{word_a}–{word_b}"
        dimension_similarity[dimension][pair_name] = score
        dimension_result_rows.append({
            "experiment": "Experiment 4 - Embedding dimension",
            "dimension": dimension, "metric": "cosine_similarity",
            "word_pair": pair_name, "value": score, "status": status,
        })

    analogy_words = ANALOGY["positive"] + ANALOGY["negative"]
    missing = [word for word in analogy_words if word not in wv]
    if missing:
        result = None
        status = "missing from vocabulary: " + ", ".join(missing)
        print("Analogy unavailable —", status)
    else:
        result = wv.most_similar(
            positive=ANALOGY["positive"], negative=ANALOGY["negative"], topn=1
        )[0]
        status = "ok"
        print(f"Analogy king - man + woman: {result[0]} ({result[1]:.6f})")
    dimension_analogy[dimension] = {"result": result, "status": status}
    dimension_result_rows.append({
        "experiment": "Experiment 4 - Embedding dimension",
        "dimension": dimension, "metric": "analogy_top1",
        "word_pair": "king - man + woman",
        "value": None if result is None else result[0],
        "status": status,
        "details": "" if result is None else f"similarity={result[1]:.6f}",
    })



Dimension 50; vocabulary size=102380
doctor – physician: 0.832808
doctor – hospital: 0.521433
cat – dog: 0.835246
Analogy king - man + woman: queen (0.783875)

Dimension 100; vocabulary size=102380
doctor – physician: 0.753575
doctor – hospital: 0.450375
cat – dog: 0.773208
Analogy king - man + woman: prince (0.679868)

Dimension 300; vocabulary size=102380
doctor – physician: 0.713918
doctor – hospital: 0.428580
cat – dog: 0.683118
Analogy king - man + woman: queen (0.605553)


## 3. Downstream task: phân loại từ theo nhóm

Dùng một bộ từ nhỏ có nhãn cố định. Từ train tạo centroid cho mỗi nhóm; từ test được gán vào centroid gần nhất theo cosine similarity. Đây là task kiểm tra nhỏ do người làm bài định nghĩa, không phải nhãn có sẵn của C4. Từ OOV được báo riêng; cùng danh sách và cách đánh giá áp dụng cho cả ba dimensions.


In [12]:
import numpy as np

DOWNSTREAM_TRAIN = {
    "medical": ["doctor", "hospital", "nurse"],
    "animal": ["cat", "dog", "bird"],
    "technology": ["computer", "software", "internet"],
}
DOWNSTREAM_TEST = {
    "medical": ["physician", "patient", "disease"],
    "animal": ["tiger", "horse", "fish"],
    "technology": ["hardware", "keyboard", "server"],
}

def cosine_for_vectors(vector_a, vector_b):
    # Tự tính cosine; xử lý vector zero để không chia cho 0.
    norm_a, norm_b = np.linalg.norm(vector_a), np.linalg.norm(vector_b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(vector_a, vector_b) / (norm_a * norm_b))

def evaluate_word_categories(wv):
    # Tạo centroid từ train words, rồi đo accuracy trên test words có trong vocabulary.
    centroids, missing_train = {}, []
    for label, words in DOWNSTREAM_TRAIN.items():
        known = [word for word in words if word in wv]
        missing_train.extend(word for word in words if word not in wv)
        if known:
            centroids[label] = np.mean([wv[word] for word in known], axis=0)

    predictions, missing_test = [], []
    for true_label, words in DOWNSTREAM_TEST.items():
        for word in words:
            if word not in wv:
                missing_test.append(word)
                continue
            if centroids:
                predicted = max(
                    centroids,
                    key=lambda label: cosine_for_vectors(wv[word], centroids[label]),
                )
                predictions.append((word, true_label, predicted))

    correct = sum(true == pred for _, true, pred in predictions)
    accuracy = correct / len(predictions) if predictions else None
    return {
        "accuracy": accuracy, "correct": correct, "evaluated": len(predictions),
        "predictions": predictions, "missing_train": missing_train,
        "missing_test": missing_test,
        "status": "ok" if predictions else "no test words in vocabulary",
    }

downstream_results = {}
for dimension, dimension_model in DIMENSION_MODELS.items():
    result = evaluate_word_categories(dimension_model.wv)
    downstream_results[dimension] = result
    score = "N/A" if result["accuracy"] is None else f"{result['accuracy']:.3f}"
    print(f"Dimension {dimension}: accuracy={score} ({result['correct']}/{result['evaluated']})")
    print("  Missing training words:", result["missing_train"] or "none")
    print("  Missing test words:", result["missing_test"] or "none")
    for word, true_label, predicted in result["predictions"]:
        print(f"  {word}: expected={true_label}, predicted={predicted}")
    dimension_result_rows.append({
        "experiment": "Experiment 4 - Embedding dimension",
        "dimension": dimension, "metric": "downstream_accuracy",
        "value": result["accuracy"], "status": result["status"],
        "details": (
            f"correct={result['correct']}; evaluated={result['evaluated']}; "
            f"missing_train={result['missing_train']}; missing_test={result['missing_test']}"
        ),
    })


Dimension 50: accuracy=1.000 (9/9)
  Missing training words: none
  Missing test words: none
  physician: expected=medical, predicted=medical
  patient: expected=medical, predicted=medical
  disease: expected=medical, predicted=medical
  tiger: expected=animal, predicted=animal
  horse: expected=animal, predicted=animal
  fish: expected=animal, predicted=animal
  hardware: expected=technology, predicted=technology
  keyboard: expected=technology, predicted=technology
  server: expected=technology, predicted=technology
Dimension 100: accuracy=1.000 (9/9)
  Missing training words: none
  Missing test words: none
  physician: expected=medical, predicted=medical
  patient: expected=medical, predicted=medical
  disease: expected=medical, predicted=medical
  tiger: expected=animal, predicted=animal
  horse: expected=animal, predicted=animal
  fish: expected=animal, predicted=animal
  hardware: expected=technology, predicted=technology
  keyboard: expected=technology, predicted=technology
  s

## 4. Bảng tổng hợp và lưu results.csv

In [13]:
from IPython.display import display, Markdown
import csv

summary_rows = [
    "| Dimension | Training time (s) | Model size (values; MiB) | Analogy top-1 | Downstream accuracy |",
    "| ---: | ---: | --- | --- | ---: |",
]
for dimension in DIMENSIONS:
    stats = dimension_stats[dimension]
    analogy = dimension_analogy[dimension]["result"]
    analogy_text = "N/A" if analogy is None else f"{analogy[0]} ({analogy[1]:.4f})"
    downstream = downstream_results[dimension]
    downstream_text = (
        "N/A" if downstream["accuracy"] is None
        else f"{downstream['accuracy']:.3f} ({downstream['correct']}/{downstream['evaluated']})"
    )
    size_text = f"{stats['embedding_values']:,}; {stats['embedding_bytes'] / (1024 ** 2):.2f}"
    summary_rows.append(
        f"| {dimension} | {stats['training_time_seconds']:.2f} | "
        f"{size_text} | {analogy_text} | {downstream_text} |"
    )
display(Markdown("\n".join(summary_rows)))

pair_rows = [
    "| Word pair | Dim 50 | Dim 100 | Dim 300 |",
    "| --- | ---: | ---: | ---: |",
]
for word_a, word_b in DIMENSION_WORD_PAIRS:
    pair_name = f"{word_a}–{word_b}"
    scores = []
    for dimension in DIMENSIONS:
        value = dimension_similarity[dimension][pair_name]
        scores.append("N/A" if value is None else f"{value:.6f}")
    pair_rows.append(f"| {pair_name} | {scores[0]} | {scores[1]} | {scores[2]} |")
display(Markdown("\n".join(pair_rows)))

# Lưu kết quả chi tiết theo metric; giữ các dòng của Experiment 1/3.
RESULTS_PATH = DATA_PATH.parent / "results.csv"
experiment_name = "Experiment 4 - Embedding dimension"
for dimension in DIMENSIONS:
    stats = dimension_stats[dimension]
    dimension_result_rows.extend([
        {
            "experiment": experiment_name, "dimension": dimension,
            "metric": "training_time_seconds",
            "value": stats["training_time_seconds"], "status": "ok",
        },
        {
            "experiment": experiment_name, "dimension": dimension,
            "metric": "embedding_vector_values",
            "value": stats["embedding_values"], "status": "ok",
            "details": "vocabulary_size × dimension; input embedding matrix only",
        },
        {
            "experiment": experiment_name, "dimension": dimension,
            "metric": "embedding_matrix_bytes",
            "value": stats["embedding_bytes"], "status": "ok",
            "details": "input embedding matrix only",
        },
    ])

new_fields = [
    "experiment", "dimension", "metric", "word_pair", "value", "status",
    "details", "window", "vector_size", "min_count", "epochs", "sg", "objective",
]
old_rows, old_fields = [], []
if RESULTS_PATH.exists():
    with RESULTS_PATH.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        old_fields, old_rows = reader.fieldnames or [], list(reader)
old_rows = [row for row in old_rows if row.get("experiment") != experiment_name]
fieldnames = list(dict.fromkeys(old_fields + new_fields))

for row in dimension_result_rows:
    row.setdefault("window", DIMENSION_WINDOW)
    row.setdefault("vector_size", row.get("dimension"))
    row.setdefault("min_count", MIN_COUNT)
    row.setdefault("epochs", EPOCHS)
    row.setdefault("sg", SG)
    row.setdefault("objective", TRAINING_OBJECTIVE)

with RESULTS_PATH.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(old_rows)
    writer.writerows(dimension_result_rows)

print("Đã lưu kết quả vào:", RESULTS_PATH)
print("Số dòng Experiment 4:", len(dimension_result_rows))


| Dimension | Training time (s) | Model size (values; MiB) | Analogy top-1 | Downstream accuracy |
| ---: | ---: | --- | --- | ---: |
| 50 | 139.50 | 5,119,000; 19.53 | queen (0.7839) | 1.000 (9/9) |
| 100 | 139.15 | 10,238,000; 39.05 | prince (0.6799) | 1.000 (9/9) |
| 300 | 230.07 | 30,714,000; 117.16 | queen (0.6056) | 1.000 (9/9) |

| Word pair | Dim 50 | Dim 100 | Dim 300 |
| --- | ---: | ---: | ---: |
| doctor–physician | 0.832808 | 0.753575 | 0.713918 |
| doctor–hospital | 0.521433 | 0.450375 | 0.428580 |
| cat–dog | 0.835246 | 0.773208 | 0.683118 |

Đã lưu kết quả vào: /home/nguyenducthang/NLP/Lab3/results.csv
Số dòng Experiment 4: 24


# Section 18 — Inspect embeddings

Các cell dưới đây dùng model Word2Vec vừa train ở Experiment 2 và tokenized_sentences từ cùng corpus. Top-5, câu và context được sinh trực tiếp khi chạy; không có kết quả điền sẵn. Chạy notebook từ đầu để model còn trong bộ nhớ.

## 18.1 Kiểm tra vocabulary và lấy Top-5 từ gần doctor

In [14]:
words_to_inspect = [
    "doctor", "hospital", "patient", "disease",
    "computer", "football", "banana",
]

print("Word nằm trong vocabulary của model:")
for word in words_to_inspect:
    print(f"{word}: {'có' if word in model.wv else 'không có'}")

if "doctor" not in model.wv:
    raise ValueError("'doctor' không có trong vocabulary; không thể gọi most_similar cho từ này.")

# Lấy trực tiếp kết quả thật từ model đã train, không tự nhập Top-5.
top5_doctor = model.wv.most_similar("doctor", topn=5)

# In kết quả thành bảng Markdown.
from IPython.display import display, Markdown

table_rows = ["| Rank | Word | Similarity |", "| ---: | --- | ---: |"]
for rank, (word, similarity) in enumerate(top5_doctor, start=1):
    table_rows.append(f"| {rank} | {word} | {similarity:.6f} |")
display(Markdown("\n".join(table_rows)))

Word nằm trong vocabulary của model:
doctor: có
hospital: có
patient: có
disease: có
computer: có
football: có
banana: có


| Rank | Word | Similarity |
| ---: | --- | ---: |
| 1 | dentist | 0.824967 |
| 2 | physician | 0.753575 |
| 3 | surgeon | 0.727624 |
| 4 | pharmacist | 0.696635 |
| 5 | veterinarian | 0.696269 |

## 18.2 Trích câu và context làm evidence

find_sentences_containing và find_contexts tìm trên toàn bộ câu đã tokenize. Context lấy tối đa model.window token ở mỗi phía. Mọi kết quả được giữ trong evidence_by_word; mặc định chỉ in 10 câu/context đầu để tránh output notebook quá dài. Đổi MAX_ITEMS_TO_PRINT = None nếu muốn in toàn bộ. Câu có đồng thời doctor và từ Top-5 được tìm riêng khi corpus có.

In [15]:
def find_sentences_containing(word, sentences=tokenized_sentences):
    # Lấy toàn bộ câu có token trùng chính xác với từ cần tìm.
    word = word.lower()
    return [sentence for sentence in sentences if word in sentence]


def find_sentences_with_both(word_a, word_b, sentences=tokenized_sentences):
    # Lấy câu có đồng thời cả hai từ để kiểm tra evidence trực tiếp.
    word_a, word_b = word_a.lower(), word_b.lower()
    return [sentence for sentence in sentences if word_a in sentence and word_b in sentence]


def find_contexts(word, sentences=tokenized_sentences, window=None):
    # Lấy các token xung quanh từ với window của Word2Vec.
    word = word.lower()
    if window is None:
        window = model.window
    contexts = []
    for sentence in sentences:
        for index, token in enumerate(sentence):
            if token == word:
                left = sentence[max(0, index - window):index]
                right = sentence[index + 1:index + window + 1]
                contexts.append(" ".join(left + [f"[{word}]"] + right))
    return contexts


MAX_ITEMS_TO_PRINT = 10  # None để in toàn bộ kết quả.
evidence_by_word = {}

for word, similarity in top5_doctor:
    evidence_by_word[word] = {
        "Word": word,
        "Similarity": float(similarity),
        "Sentences containing doctor": find_sentences_containing("doctor"),
        "Sentences containing the word": find_sentences_containing(word),
        "Sentences containing both doctor and this word": find_sentences_with_both("doctor", word),
        "Contexts of doctor": find_contexts("doctor"),
        "Contexts of this word": find_contexts(word),
    }
    evidence = evidence_by_word[word]
    print("\n" + "-" * 80)
    print(f"Word: {word}")
    print(f"Similarity: {evidence['Similarity']:.6f}")
    for label in ["Sentences containing doctor", "Sentences containing the word", "Sentences containing both doctor and this word", "Contexts of doctor", "Contexts of this word"]:
        items = evidence[label]
        shown_items = items if MAX_ITEMS_TO_PRINT is None else items[:MAX_ITEMS_TO_PRINT]
        print(f"{label} ({len(items)} found; showing {len(shown_items)}):")
        for item in shown_items:
            print("-", " ".join(item) if isinstance(item, list) else item)


--------------------------------------------------------------------------------
Word: dentist
Similarity: 0.824967
Sentences containing doctor (740 found; showing 10):
- inspector beck h division deposed that having sent for the doctor he gave orders to prevent any persons leaving the court and he directed officers to make a search
- assuming leadership roles at all stages of their training and career is a progressively vital component of the definition of a good doctor
- should your skincare clinic not have an overseeing doctor chances are that you are not receiving the appropriate numbing cream only a 2 lidocaine which can make this treatment uncomfortable
- need to make a doctor appointment for tetralogy of fallot tof this week
- i'm not a doctor but that's my diagnosis
- she was diagnosed in 2011 when her doctor noticed something in her blood work after her annual checkup
- about a year ago her doctor told her she was cured
- the doctor said that it was the type of fracture that 


--------------------------------------------------------------------------------
Word: physician
Similarity: 0.753575
Sentences containing doctor (740 found; showing 10):
- inspector beck h division deposed that having sent for the doctor he gave orders to prevent any persons leaving the court and he directed officers to make a search
- assuming leadership roles at all stages of their training and career is a progressively vital component of the definition of a good doctor
- should your skincare clinic not have an overseeing doctor chances are that you are not receiving the appropriate numbing cream only a 2 lidocaine which can make this treatment uncomfortable
- need to make a doctor appointment for tetralogy of fallot tof this week
- i'm not a doctor but that's my diagnosis
- she was diagnosed in 2011 when her doctor noticed something in her blood work after her annual checkup
- about a year ago her doctor told her she was cured
- the doctor said that it was the type of fracture tha


--------------------------------------------------------------------------------
Word: surgeon
Similarity: 0.727624
Sentences containing doctor (740 found; showing 10):
- inspector beck h division deposed that having sent for the doctor he gave orders to prevent any persons leaving the court and he directed officers to make a search
- assuming leadership roles at all stages of their training and career is a progressively vital component of the definition of a good doctor
- should your skincare clinic not have an overseeing doctor chances are that you are not receiving the appropriate numbing cream only a 2 lidocaine which can make this treatment uncomfortable
- need to make a doctor appointment for tetralogy of fallot tof this week
- i'm not a doctor but that's my diagnosis
- she was diagnosed in 2011 when her doctor noticed something in her blood work after her annual checkup
- about a year ago her doctor told her she was cured
- the doctor said that it was the type of fracture that 


--------------------------------------------------------------------------------
Word: pharmacist
Similarity: 0.696635
Sentences containing doctor (740 found; showing 10):
- inspector beck h division deposed that having sent for the doctor he gave orders to prevent any persons leaving the court and he directed officers to make a search
- assuming leadership roles at all stages of their training and career is a progressively vital component of the definition of a good doctor
- should your skincare clinic not have an overseeing doctor chances are that you are not receiving the appropriate numbing cream only a 2 lidocaine which can make this treatment uncomfortable
- need to make a doctor appointment for tetralogy of fallot tof this week
- i'm not a doctor but that's my diagnosis
- she was diagnosed in 2011 when her doctor noticed something in her blood work after her annual checkup
- about a year ago her doctor told her she was cured
- the doctor said that it was the type of fracture th


--------------------------------------------------------------------------------
Word: veterinarian
Similarity: 0.696269
Sentences containing doctor (740 found; showing 10):
- inspector beck h division deposed that having sent for the doctor he gave orders to prevent any persons leaving the court and he directed officers to make a search
- assuming leadership roles at all stages of their training and career is a progressively vital component of the definition of a good doctor
- should your skincare clinic not have an overseeing doctor chances are that you are not receiving the appropriate numbing cream only a 2 lidocaine which can make this treatment uncomfortable
- need to make a doctor appointment for tetralogy of fallot tof this week
- i'm not a doctor but that's my diagnosis
- she was diagnosed in 2011 when her doctor noticed something in her blood work after her annual checkup
- about a year ago her doctor told her she was cured
- the doctor said that it was the type of fracture 

## Evidence từ lần chạy — ví dụ dentist

- Word: dentist
- Similarity: 0.824967
- Sentences containing the word:
  - however it wasn’t a doctor who diagnosed him but a dentist
- Contexts of doctor:
  - however it wasn t a [doctor] who diagnosed him but a
- Contexts of this word:
  - who diagnosed him but a [dentist]
- My analysis: Từ `dentist` có độ tương đồng khá cao với `doctor` (0.824967). Trong corpus, hai từ này xuất hiện trong cùng một câu và ở rất gần nhau: “it wasn’t a doctor who diagnosed him but a dentist”. Đây là bằng chứng ngữ cảnh trực tiếp cho thấy `doctor` và `dentist` có thể xuất hiện trong những môi trường ngôn ngữ tương tự.

Ngoài ra, trong câu này cả hai từ đều liên quan đến hành động `diagnosed`, tức là cùng xuất hiện trong một ngữ cảnh về chẩn đoán. Vì vậy, model có thể học được các vector của `doctor` và `dentist` nằm gần nhau trong không gian embedding.

Tuy nhiên, kết quả này chỉ dựa trên ngữ cảnh xuất hiện trong corpus hiện tại, nên similarity cao không có nghĩa là hai từ hoàn toàn đồng nghĩa.

# Section 21 — Evaluation: Word similarity

Phần này dùng model đã train ở Experiment 2 làm baseline (vector_size=100, window=5). Với mỗi cặp, code kiểm tra vocabulary rồi mới tính cosine similarity. Cặp có từ OOV được báo riêng, không thay từ và không tạo score.


## 21.1 Kiểm tra vocabulary và tính cosine similarity

In [16]:
EVALUATION_WORD_PAIRS = [
    ("doctor", "physician"),
    ("car", "automobile"),
    ("king", "queen"),
    ("cat", "dog"),
    ("computer", "banana"),
]

# Dùng model baseline của Experiment 2, không dùng nhầm model khác.
EVAL_MODEL = model
EVAL_MODEL_NAME = "Experiment 2 baseline (vector_size=100, window=5)"
similarity_results = []

for word1, word2 in EVALUATION_WORD_PAIRS:
    missing_words = [word for word in (word1, word2) if word not in EVAL_MODEL.wv]

    if missing_words:
        result = {
            "word1": word1, "word2": word2, "similarity": None,
            "status": "missing from vocabulary: " + ", ".join(missing_words),
            "missing_words": missing_words,
        }
        print(f"{word1} – {word2}: không tính; thiếu {', '.join(missing_words)}")
    else:
        score = float(EVAL_MODEL.wv.similarity(word1, word2))
        result = {
            "word1": word1, "word2": word2, "similarity": score,
            "status": "ok", "missing_words": [],
        }
        print(f"{word1} – {word2}: similarity={score:.6f}")

    similarity_results.append(result)


doctor – physician: similarity=0.753575
car – automobile: similarity=0.624797
king – queen: similarity=0.726507
cat – dog: similarity=0.773208
computer – banana: similarity=-0.230996


## 21.2 Bảng similarity và xếp hạng giảm dần

In [17]:
from IPython.display import display, Markdown

# Cặp OOV hiện N/A; không tự tạo score.
result_table = [
    "| Word pair | Similarity | Status |",
    "| --- | ---: | --- |",
]
for result in similarity_results:
    pair = f"{result['word1']} – {result['word2']}"
    score = "N/A" if result["similarity"] is None else f"{result['similarity']:.6f}"
    status = result["status"]
    result_table.append(f"| {pair} | {score} | {status} |")
display(Markdown("\n".join(result_table)))

# Chỉ xếp hạng các cặp có score thực; OOV được loại khỏi ranking.
ranked_results = sorted(
    (item for item in similarity_results if item["similarity"] is not None),
    key=lambda item: item["similarity"], reverse=True,
)
ranking_table = [
    "| Rank | Word pair | Similarity |",
    "| ---: | --- | ---: |",
]
for rank, result in enumerate(ranked_results, start=1):
    pair = f"{result['word1']} – {result['word2']}"
    ranking_table.append(f"| {rank} | {pair} | {result['similarity']:.6f} |")
display(Markdown("\n".join(ranking_table)))

if len(ranked_results) < len(EVALUATION_WORD_PAIRS):
    print(
        f"Ranking gồm {len(ranked_results)} cặp hợp lệ; "
        f"{len(EVALUATION_WORD_PAIRS) - len(ranked_results)} cặp thiếu vocabulary."
    )


| Word pair | Similarity | Status |
| --- | ---: | --- |
| doctor – physician | 0.753575 | ok |
| car – automobile | 0.624797 | ok |
| king – queen | 0.726507 | ok |
| cat – dog | 0.773208 | ok |
| computer – banana | -0.230996 | ok |

| Rank | Word pair | Similarity |
| ---: | --- | ---: |
| 1 | cat – dog | 0.773208 |
| 2 | doctor – physician | 0.753575 |
| 3 | king – queen | 0.726507 |
| 4 | car – automobile | 0.624797 |
| 5 | computer – banana | -0.230996 |

## 21.3 Tìm câu/context làm evidence

In [18]:
def find_pair_corpus_evidence(word1, word2, sentences=tokenized_sentences, window=None):
    # Tìm câu có đồng thời hai từ và lấy context theo window của model.
    word1, word2 = word1.lower(), word2.lower()
    if window is None:
        window = EVAL_MODEL.window

    both_sentences, contexts1, contexts2 = [], [], []
    for sentence in sentences:
        if word1 not in sentence or word2 not in sentence:
            continue
        both_sentences.append(" ".join(sentence))
        for word, output in ((word1, contexts1), (word2, contexts2)):
            for index, token in enumerate(sentence):
                if token == word:
                    left = sentence[max(0, index - window):index]
                    right = sentence[index + 1:index + window + 1]
                    output.append(" ".join(left + [f"[{word}]"] + right))

    return {
        "sentences_with_both": both_sentences,
        "contexts_word1": contexts1,
        "contexts_word2": contexts2,
    }


# Toàn bộ kết quả được giữ trong pair_evidence.
# Đặt None để in mọi kết quả; mặc định chỉ in tối đa 10 ví dụ mỗi trường.
MAX_EVIDENCE_TO_PRINT = 10
pair_evidence = {}
for result in similarity_results:
    word1, word2 = result["word1"], result["word2"]
    evidence = find_pair_corpus_evidence(word1, word2)
    pair_evidence[(word1, word2)] = evidence

    print(f"\n{word1} – {word2}")
    for label, key in (
        ("Sentences containing both words", "sentences_with_both"),
        (f"Contexts of {word1}", "contexts_word1"),
        (f"Contexts of {word2}", "contexts_word2"),
    ):
        items = evidence[key]
        shown = items if MAX_EVIDENCE_TO_PRINT is None else items[:MAX_EVIDENCE_TO_PRINT]
        print(f"{label}: {len(items)} found; showing {len(shown)}")
        for item in shown:
            print(" -", item)



doctor – physician
Sentences containing both words: 3 found; showing 3
 - to have been the victim of a physician that gave you a prescription that only made you more ill or had side effects that have led to severe depression or a life threatening reaction to have been under the knife of a surgeon who removed the wrong organ to have been diagnosed by a doctor not competent to carry out the task is a terrible and frightening thing
 - doctor jim andrews noted sports medicine physician has had his research team working on identifying the proper biomechanical skills for several years
 - recent research has found that it can be difficult to find a concierge doctor or a direct primary care physician rural areas such as idaho north dakota south dakota louisiana and mississippi
Contexts of doctor: 3 found; showing 3
 - have been diagnosed by a [doctor] not competent to carry out
 - [doctor] jim andrews noted sports medicine
 - difficult to find a concierge [doctor] or a direct primary care
Con


car – automobile
Sentences containing both words: 11 found; showing 10
 - if your ne auto liability only insurance online however often the same age vehicle usage and car insurance is something every person you know how to find an automobile cover for the travel trip by your surroundings
 - a car or car is a wheeled automobile used for transport
 - our wedding automobile hire services consist of a red carpet grand entryway full wedding event decorations for the car complimentary champagne with all stretch limousine plans and tyre covers to keep the bride to be’s lovely dress in beautiful condition
 - a when you loan your car to a friend or an associate he or she will be covered under your automobile insurance policy
 - the cuteness of the car aided by a full body wrap created a clever personality for this automobile
 - we do our ideal to make it simple for you to find a utilised truck that greatest suits your style at the used automobile dealerships of the automobile company warsaw an


king – queen
Sentences containing both words: 36 found; showing 10
 - cloth headboard upholstered headboard king bookcase headboard queen queen size fabric headboard french headboard white padded headboard queen upholstered headboard queen diy
 - truly a slipper made for a princess or queen i love the king and i as well
 - this image gallery named as slumber solutions gel 2 inch queen king cal king size might be ideal solution for your decoration
 - in the event you would like to create your slumber solutions gel 2 inch queen king cal king size like a focal level you then ought to make the additional household furniture to appear less flashy and elaborate since the desk
 - however of course you will find different kinds of persons and distinct kinds of persons ordinarily want various sorts of slumber solutions gel 2 inch queen king cal king size
 - believe it or notmany people always have a problem in picking out the ideal slumber solutions gel 2 inch queen king cal king size
 - those


cat – dog
Sentences containing both words: 43 found; showing 10
 - mostly it’s a feeling of ultimate peace and an over whelming calmness that comes over me at different times during the day when god allows me glimpses of his love for me when my dog and cat are doing something cute together when i see the birds eating the seed i put out for them when a certain little girl comes over to the house and calls me grandma
 - pet rabbits r b ming popular t h v indoor pets a dog r cat
 - on january 4th 1999 the belfast telegraph or the belly telly as we have heard it referred to told us that the big cat is a dog and that according to the ruc sightings of a large puma like cat in tyrone over the weekend are probably shaggy dog stories
 - do not put a chinchilla's cage in a spot where a dog cat or anything else can harm or injure it
 - we test for the most common environmental allergens that cause an ige reaction to form on the surface of the skin including dust pollens ragweed mold cat and dog 


computer – banana
Sentences containing both words: 0 found; showing 0
Contexts of computer: 0 found; showing 0
Contexts of banana: 0 found; showing 0


## 21.4 Lưu kết quả định lượng vào results.csv

In [19]:
import csv

RESULTS_PATH = DATA_PATH.parent / "results.csv"
EXPERIMENT_NAME = "Section 21 - Word similarity"

new_result_rows = []
for result in similarity_results:
    new_result_rows.append({
        "experiment": EXPERIMENT_NAME,
        "model": EVAL_MODEL_NAME,
        "word_pair": f"{result['word1']}–{result['word2']}",
        "word1": result["word1"],
        "word2": result["word2"],
        "similarity": result["similarity"],
        "status": result["status"],
        "missing_words": ", ".join(result["missing_words"]),
        "vector_size": EVAL_MODEL.vector_size,
        "window": EVAL_MODEL.window,
        "min_count": EVAL_MODEL.min_count,
        "epochs": EPOCHS,
        "sg": EVAL_MODEL.sg,
        "objective": TRAINING_OBJECTIVE,
    })

new_fields = [
    "experiment", "model", "word_pair", "word1", "word2", "similarity",
    "status", "missing_words", "vector_size", "window", "min_count",
    "epochs", "sg", "objective",
]
old_rows, old_fields = [], []
if RESULTS_PATH.exists():
    with RESULTS_PATH.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        old_fields, old_rows = reader.fieldnames or [], list(reader)

# Thay riêng kết quả Section 21 cũ, không ảnh hưởng các thí nghiệm khác.
old_rows = [row for row in old_rows if row.get("experiment") != EXPERIMENT_NAME]
fieldnames = list(dict.fromkeys(old_fields + new_fields))
with RESULTS_PATH.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(old_rows)
    writer.writerows(new_result_rows)

print("Đã lưu kết quả vào:", RESULTS_PATH)
print("Số cặp đã ghi:", len(new_result_rows))


Đã lưu kết quả vào: /home/nguyenducthang/NLP/Lab3/results.csv
Số cặp đã ghi: 5


- **Expected ranking:**  
  1. doctor – physician  
  2. car – automobile  
  3. king – queen  
  4. cat – dog  
  5. computer – banana  

- **Observed ranking:**  
  1. cat – dog: 0.773208  
  2. doctor – physician: 0.753575  
  3. king – queen: 0.726507  
  4. car – automobile: 0.624797  
  5. computer – banana: -0.230996  

- **Matches intuition:**  
  Nhìn chung kết quả phù hợp với trực giác vì các cặp có quan hệ ngữ nghĩa như `doctor–physician`, `king–queen`, `car–automobile` và `cat–dog` đều có similarity dương tương đối cao. Trong khi đó, `computer–banana` là hai từ gần như không liên quan về nghĩa nên có similarity âm và đứng cuối bảng.

- **Unexpected cases:**  
  Trường hợp đáng chú ý nhất là `cat–dog` có similarity cao nhất (0.773208), thậm chí cao hơn `doctor–physician` (0.753575), mặc dù theo trực giác `doctor` và `physician` gần như đồng nghĩa. Ngoài ra, `car–automobile` cũng là hai từ gần nghĩa nhưng similarity chỉ đạt 0.624797 và đứng sau `king–queen`. Điều này cho thấy similarity của Word2Vec không chỉ phụ thuộc vào nghĩa từ điển mà còn phụ thuộc vào cách các từ xuất hiện trong corpus.

- **Possible evidence from corpus:**  
  Cần kiểm tra các câu và context thực tế chứa `cat`, `dog`, `doctor`, `physician`, `car` và `automobile`. Nếu `cat` và `dog` thường xuất hiện trong các context tương tự hơn, còn `doctor–physician` hoặc `car–automobile` xuất hiện ít hơn hoặc trong các context khác nhau, điều đó có thể giải thích thứ hạng quan sát được. Phần này nên trích trực tiếp các câu từ corpus để làm bằng chứng.

# Section 22 — Evaluation: Word analogy

Các cell này dùng model baseline đã train ở Experiment 2. Kết quả Top-5 được truy vấn trực tiếp từ model; nếu từ đầu vào thiếu thì phép toán tương ứng sẽ không chạy.


## 22.1 Kiểm tra vocabulary

In [20]:
ANALOGY_WORDS_TO_CHECK = [
    "king", "man", "woman", "queen",
    "paris", "france", "italy", "rome",
]
ANALOGY_MODEL = model

print("Vocabulary check trong model Experiment 2:")
for word in ANALOGY_WORDS_TO_CHECK:
    print(f"{word}: {'có' if word in ANALOGY_MODEL.wv else 'không có'}")


Vocabulary check trong model Experiment 2:
king: có
man: có
woman: có
queen: có
paris: có
france: có
italy: có
rome: có


## 22.2 king - man + woman

In [21]:
from IPython.display import display, Markdown

king_inputs = ["king", "woman", "man"]
missing_king_inputs = [word for word in king_inputs if word not in ANALOGY_MODEL.wv]

if missing_king_inputs:
    king_analogy_results = None
    print("Không chạy được king - man + woman; thiếu:", ", ".join(missing_king_inputs))
else:
    # Lấy Top-5 trực tiếp từ Word2Vec, không tự nhập kết quả.
    king_analogy_results = ANALOGY_MODEL.wv.most_similar(
        positive=["king", "woman"],
        negative=["man"],
        topn=5,
    )

    rows = [
        "| Rank | Word | Similarity |",
        "| ---: | --- | ---: |",
    ]
    for rank, (word, similarity) in enumerate(king_analogy_results, start=1):
        rows.append(f"| {rank} | {word} | {similarity:.6f} |")
    display(Markdown("\n".join(rows)))

    queen_result = next(
        ((rank, word, score)
         for rank, (word, score) in enumerate(king_analogy_results, start=1)
         if word == "queen"),
        None,
    )
    if queen_result is not None:
        rank, _, score = queen_result
        print(f"queen có trong Top-5: có; rank={rank}; similarity={score:.6f}")
    elif "queen" not in ANALOGY_MODEL.wv:
        print("queen không có trong vocabulary nên không thể xuất hiện trong kết quả.")
    else:
        print("queen có trong vocabulary nhưng không xuất hiện trong Top-5.")


| Rank | Word | Similarity |
| ---: | --- | ---: |
| 1 | prince | 0.679868 |
| 2 | queen | 0.653382 |
| 3 | charles | 0.617888 |
| 4 | teresa | 0.594969 |
| 5 | albert | 0.593760 |

queen có trong Top-5: có; rank=2; similarity=0.653382


## 22.3 Paris - France + Italy

In [22]:
paris_inputs = ["paris", "italy", "france"]
missing_paris_inputs = [word for word in paris_inputs if word not in ANALOGY_MODEL.wv]

if missing_paris_inputs:
    paris_analogy_results = None
    print("Không chạy được Paris - France + Italy; thiếu:", ", ".join(missing_paris_inputs))
else:
    paris_analogy_results = ANALOGY_MODEL.wv.most_similar(
        positive=["paris", "italy"],
        negative=["france"],
        topn=5,
    )

    rows = [
        "| Rank | Word | Similarity |",
        "| ---: | --- | ---: |",
    ]
    for rank, (word, similarity) in enumerate(paris_analogy_results, start=1):
        rows.append(f"| {rank} | {word} | {similarity:.6f} |")
    display(Markdown("\n".join(rows)))

    rome_result = next(
        ((rank, word, score)
         for rank, (word, score) in enumerate(paris_analogy_results, start=1)
         if word == "rome"),
        None,
    )
    if rome_result is not None:
        rank, _, score = rome_result
        print(f"rome có trong Top-5: có; rank={rank}; similarity={score:.6f}")
    elif "rome" not in ANALOGY_MODEL.wv:
        print("rome không có trong vocabulary; không thể xuất hiện trong kết quả.")
    else:
        print("rome có trong vocabulary nhưng không xuất hiện trong Top-5.")


| Rank | Word | Similarity |
| ---: | --- | ---: |
| 1 | venice | 0.686559 |
| 2 | terre | 0.675629 |
| 3 | bali | 0.650562 |
| 4 | bucharest | 0.643385 |
| 5 | istanbul | 0.637238 |

rome có trong vocabulary nhưng không xuất hiện trong Top-5.


## Ý tưởng hình học

Embedding đặt mỗi từ tại một vector trong không gian. Một số quan hệ có thể tạo ra các hướng/độ lệch tương đối nhất quán; phép cộng/trừ vector khai thác các pattern hình học-thống kê này. Đây không phải bằng chứng về human-like understanding.

## Kết quả và template tự hoàn thiện

- Analogy: king - man + woman
- Expected: queen
- Observed Top-5:
  1. prince: 0.679868
  2. queen: 0.653382
  3. charles: 0.617888
  4. teresa: 0.594969
  5. albert: 0.593760
- Rank of expected word: 2
- Similarity: 0.653382
- Explanation: Với phép toán king - man + woman, từ kỳ vọng là queen. Kết quả cho thấy queen đứng thứ 2 với similarity = 0.653382, chỉ sau prince = 0.679868. Điều này cho thấy embedding đã học được một phần pattern liên quan giữa các từ về giới tính và hoàng gia. Phép cộng/trừ vector có thể khai thác các hướng tương đối trong không gian embedding, nên vector mới nằm khá gần queen. Tuy nhiên, kết quả không hoàn toàn chính xác vì từ đứng đầu lại là prince.

Với phép thử Paris - France + Italy, từ kỳ vọng là Rome, nhưng Rome không xuất hiện trong Top-5. Điều này cho thấy pattern quan hệ “thủ đô – quốc gia” chưa được model học đủ ổn định trong corpus hiện tại.
- Limitation: Word analogy không phải lúc nào cũng cho kết quả đúng. Chất lượng phụ thuộc mạnh vào corpus, tần suất xuất hiện của từ, context, kích thước embedding và quá trình huấn luyện. Nếu một quan hệ không xuất hiện đủ thường xuyên hoặc không có context nhất quán, vector arithmetic có thể trả về các từ liên quan nhưng không phải đáp án kỳ vọng.

Ngoài ra, việc king - man + woman cho kết quả gần queen chỉ cho thấy một pattern hình học-thống kê trong không gian vector. Nó không chứng minh rằng Word2Vec thực sự “hiểu” các khái niệm như giới tính, hoàng gia, quốc gia hay thủ đô giống con người.
Với phép thử Paris - France + Italy, Top-5 thực tế là venice (0.686559), terre (0.675629), bali (0.650562), bucharest (0.643385), istanbul (0.637238). Rome có trong vocabulary nhưng không nằm trong Top-5.


# Section 24 — Application: Semantic Search

Dùng Word2Vec baseline của Experiment 2. Tập tìm kiếm là mẫu cố định 200 documents lấy từ cùng C4 JSONL; documents được lowercase và tokenize bằng đúng các regex của Experiment 2. Không dùng sentence-transformers hay pretrained embeddings.


## Documents

In [23]:
import random

SEARCH_SAMPLE_SIZE = 200
SEARCH_SAMPLE_SEED = 24
search_rng = random.Random(SEARCH_SAMPLE_SEED)
search_documents = []
eligible_documents_seen = 0

# Đọc lại đúng corpus đã dùng trước đó; reservoir sampling lấy mẫu đều mà không giữ 30K docs trong RAM.
with DATA_PATH.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if line_number > MAX_DOCUMENTS:
            break
        if not line.strip():
            continue

        record = json.loads(line)
        raw_text = record.get("text", "") if isinstance(record, dict) else ""
        if not isinstance(raw_text, str) or not raw_text.strip():
            continue

        lowered_text = raw_text.lower()
        document_tokens = []
        for sentence_chunk in SENTENCE_SPLIT_PATTERN.split(lowered_text):
            document_tokens.extend(WORD_PATTERN.findall(sentence_chunk))
        if not document_tokens:
            continue

        candidate = {
            "doc_id": line_number,
            "text": raw_text,
            "tokens": document_tokens,
        }
        eligible_documents_seen += 1

        if len(search_documents) < SEARCH_SAMPLE_SIZE:
            search_documents.append(candidate)
        else:
            replace_index = search_rng.randrange(eligible_documents_seen)
            if replace_index < SEARCH_SAMPLE_SIZE:
                search_documents[replace_index] = candidate

print("Corpus:", DATA_PATH)
print("Documents đủ token:", eligible_documents_seen)
print("Documents được lấy mẫu:", len(search_documents))
print("Sample seed:", SEARCH_SAMPLE_SEED)
if search_documents:
    print("Document mẫu:", search_documents[0]["doc_id"])
    print(search_documents[0]["text"][:300])


Corpus: /home/nguyenducthang/NLP/Lab3/c4-train.00000-of-01024-30K.json
Documents đủ token: 30000
Documents được lấy mẫu: 200
Sample seed: 24
Document mẫu: 17656
On December 30, 2016, the revised Order of Celebrating Matrimony becomes mandatory in U.S. dioceses.
The Rite of Marriage, which served U.S. Catholics from 1970 to 2016, has given way to the Order of Celebrating Matrimony. The word “matrimony” has replaced “marriage” in order “to highlight the book 


## Query preprocessing

In [24]:
SEARCH_QUERY = "medical treatment"
query_words = WORD_PATTERN.findall(SEARCH_QUERY.lower())

print("Query gốc:", SEARCH_QUERY)
print("Query words:", query_words)
print("Vocabulary status:")
for word in query_words:
    print(f"{word}: {'có' if word in model.wv else 'không có'}")


Query gốc: medical treatment
Query words: ['medical', 'treatment']
Vocabulary status:
medical: có
treatment: có


## Query embedding

In [25]:
import numpy as np

def get_sentence_vector(tokens, model):
    # Lấy trung bình vector của các token nằm trong Word2Vec vocabulary.
    valid_vectors = [model.wv[token] for token in tokens if token in model.wv]
    if not valid_vectors:
        # Vector zero biểu thị không có token nào dùng được.
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(valid_vectors, axis=0)

valid_query_words = [word for word in query_words if word in model.wv]
missing_query_words = [word for word in query_words if word not in model.wv]
query_vector = get_sentence_vector(query_words, model)

print("Query words dùng để tạo vector:", valid_query_words or "không có")
print("Query words OOV:", missing_query_words or "không có")
print("Query vector shape:", query_vector.shape)
print("Query vector zero:", bool(np.linalg.norm(query_vector) == 0))


Query words dùng để tạo vector: ['medical', 'treatment']
Query words OOV: không có
Query vector shape: (100,)
Query vector zero: False


## Document embeddings

In [26]:
# Tạo vector trung bình cho mỗi document trong mẫu.
document_vectors = {}
documents_without_vocabulary_words = []

for document in search_documents:
    vector = get_sentence_vector(document["tokens"], model)
    if np.linalg.norm(vector) == 0:
        documents_without_vocabulary_words.append(document["doc_id"])
    else:
        document_vectors[document["doc_id"]] = vector

print("Document có vector:", len(document_vectors))
print("Document không có từ nào trong vocabulary:", len(documents_without_vocabulary_words))


Document có vector: 200
Document không có từ nào trong vocabulary: 0


## Cosine similarity

In [27]:
def cosine_similarity(a, b):
    # Tính cosine similarity và trả 0 nếu có zero vector để tránh chia cho 0.
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))


print("Zero-vector check:", cosine_similarity(
    np.zeros(model.vector_size), np.ones(model.vector_size)
))


Zero-vector check: 0.0


## Ranking

In [28]:
def semantic_search(query, documents, model, top_k=5):
    # Vector hóa query và từng document, sau đó sắp xếp cosine similarity giảm dần.
    query_tokens = WORD_PATTERN.findall(query.lower())
    known_query_tokens = [token for token in query_tokens if token in model.wv]
    unknown_query_tokens = [token for token in query_tokens if token not in model.wv]
    query_vector = get_sentence_vector(query_tokens, model)

    if not known_query_tokens or np.linalg.norm(query_vector) == 0:
        return {
            "results": [],
            "query_tokens": query_tokens,
            "known_query_tokens": known_query_tokens,
            "unknown_query_tokens": unknown_query_tokens,
            "documents_skipped": len(documents),
            "reason": "query không có từ hợp lệ trong vocabulary",
        }

    results = []
    skipped_documents = 0
    for document in documents:
        document_vector = get_sentence_vector(document["tokens"], model)
        if np.linalg.norm(document_vector) == 0:
            skipped_documents += 1
            continue
        results.append({
            "doc_id": document["doc_id"],
            "text": document["text"],
            "similarity": cosine_similarity(query_vector, document_vector),
        })

    results.sort(key=lambda item: item["similarity"], reverse=True)
    return {
        "results": results[:top_k],
        "query_tokens": query_tokens,
        "known_query_tokens": known_query_tokens,
        "unknown_query_tokens": unknown_query_tokens,
        "documents_skipped": skipped_documents,
        "reason": None,
    }


search_output = semantic_search(SEARCH_QUERY, search_documents, model, top_k=5)
print("Query tokens:", search_output["query_tokens"])
print("Known query words:", search_output["known_query_tokens"] or "none")
print("OOV query words:", search_output["unknown_query_tokens"] or "none")
print("Documents skipped (no vocabulary tokens):", search_output["documents_skipped"])
if search_output["reason"]:
    print("Search status:", search_output["reason"])


Query tokens: ['medical', 'treatment']
Known query words: ['medical', 'treatment']
OOV query words: none
Documents skipped (no vocabulary tokens): 0


## Results

In [29]:
from IPython.display import display, Markdown

search_table = [
    "| Rank | Document | Similarity |",
    "| ---: | --- | ---: |",
]
for rank, result in enumerate(search_output["results"], start=1):
    # Rút gọn preview để bảng đọc được; nội dung gốc vẫn có trong search_output.
    preview = " ".join(result["text"].split())[:350].replace("|", " ")
    search_table.append(
        f"| {rank} (D{result['doc_id']}) | {preview} | {result['similarity']:.6f} |"
    )
display(Markdown("\n".join(search_table)))

# Truy vấn các từ gần từng query word trực tiếp từ Word2Vec.
for query_word in search_output["known_query_tokens"]:
    print(f"\nRelated words for {query_word}:")
    related = model.wv.most_similar(query_word, topn=5)
    related_table = [
        "| Rank | Related word | Similarity |",
        "| ---: | --- | ---: |",
    ]
    for rank, (word, score) in enumerate(related, start=1):
        related_table.append(f"| {rank} | {word} | {score:.6f} |")
    display(Markdown("\n".join(related_table)))


| Rank | Document | Similarity |
| ---: | --- | ---: |
| 1 (D1127) | An Advanced Practice Provider career with UT Southwestern Medical Center is your chance to be part of an extraordinary team. We are a nationally recognized, academic medical center, home to dedicated health care professionals on a mission to improve the health care in our community, our nation and the world through education, innovation and compass | 0.601926 |
| 2 (D8156) | India Dental World helps you connect with the best Dentist from leading Dental Clinic in Gadital, Pune. Get complete clinic’s information such as address, phone number, location, ratings and other details for Gadital Dental Clinic. Book online appointment with experience dentist and dental consultants in Gadital, Pune with ease and get a healthy sm | 0.578655 |
| 3 (D28056) | Basel, Switzerland, January 03, 2018 - Basilea Pharmaceutica Ltd. (SIX: BSLN) announced today the initiation of a phase 1 study conducted under its clinical study agreement with the Adult Brain Tumor Consortium (ABTC) in the U.S. The first patient has been treated in this open-label dose-escalation study to determine the safety and tolerability of  | 0.561281 |
| 4 (D9079) | Speech and language pathologists in Chicago IL for evaluation and treatment of expressive and receptive language, articulation/phonology, oral motor, feeding, social pragmatic skills and sensory integration. Mary has been practicing for over 20 years and is co-owner of Oak Tree Developmental Center. She has worked within the school and private prac | 0.558173 |
| 5 (D9418) | The California Institute for Behavioral Health Solutions (CIBHS) work in the area of “evidence-based practices” began in 2001 with the CIBHS Center for Child and Family Services report and training Evidence-based Practices in Mental Health Services for Foster Youth. This has evolved to become an agency–wide effort designated the Values-Driven Evide | 0.514550 |


Related words for medical:


| Rank | Related word | Similarity |
| ---: | --- | ---: |
| 1 | nursing | 0.686916 |
| 2 | healthcare | 0.682572 |
| 3 | counselling | 0.681684 |
| 4 | rehabilitation | 0.669353 |
| 5 | counseling | 0.668980 |


Related words for treatment:


| Rank | Related word | Similarity |
| ---: | --- | ---: |
| 1 | treatments | 0.773023 |
| 2 | diagnosis | 0.762728 |
| 3 | medication | 0.744999 |
| 4 | surgical | 0.742550 |
| 5 | surgery | 0.728760 |

### Technical notes

Với query, code lấy trung bình embedding của các query words có trong vocabulary. Với mỗi document, code lấy trung bình embedding của các token hợp lệ rồi tính cosine với query vector. Document không có token hợp lệ bị bỏ qua; zero vector được xử lý riêng để tránh chia cho 0.

### LAB 01 và LAB 03

- **LAB 01:** query và document được biểu diễn bằng TF-IDF dựa trên trọng số/tần suất từ trong corpus; cosine dùng để xếp hạng theo lexical weighting.
- **LAB 03:** query và document được biểu diễn bằng trung bình Word2Vec embeddings; cosine so sánh hướng các vector đã học từ ngữ cảnh. Có thể lấy thêm từ gần query trong vector space.

Hãy tự xem output để đánh giá kết quả; phần này không điền sẵn interpretation.
